# BioSYM Collocation Step-by-Step Walkthrough

This notebook provides a detailed, step-by-step walkthrough of the optimal control problem (OCP) collocation setup for a walking simulation in **BioSYM**. 

You will inspect how configurations are parsed, models are loaded, states are structured under the new **lean state vector** architecture, and objectives/constraints are evaluated and compiled using **JAX**.

In [1]:
import os
import sys
import numpy as np
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt

# Ensure repo root path is set correctly to import biosym
cwd = os.getcwd()
print(f"Current working directory: {cwd}")
if cwd not in sys.path:
    sys.path.insert(0, cwd)

from biosym.model.model import load_model
from biosym.ocp import collocation
from biosym.utils import states, read_mot
os.chdir('..')

Current working directory: /Users/markusgambietz/PhD/01_Python_Projects/biosym/dorschky_demo


## 1. Load the Model and Configuration

We load the YAML configuration file for the walking initial guess (`walking2d_initial_guess.yaml`). This file defines:
* The model path (`tests/models/gait2d/gait2d.yaml`)
* Optimization settings (number of nodes $N = 100$, tolerance, solver iterations)
* Objective terms (`track_angles`, `effort_term`, `track_grf`)
* Constraints (`dynamics`, `periodicity`)
* Bounds on time duration, speed, coordinates, velocities, and actuator inputs.

In [2]:
walking_yaml = "dorschky_demo/configs/walking2d_initial_guess.yaml"

# Load the walking collocation problem
ocp = collocation.Collocation(walking_yaml)
print(f"Collocation OCP name: {ocp.settings.get('name')}")
print(f"Number of nodes: {ocp.settings['nnodes']}")
print(f"Model DOF count: {ocp.model.coordinates['n']}")
print(f"Coordinates: {ocp.model.coordinates['names']}")

Loading model from cache: d7c5c9df03908e13b72fffad79112adf8b0b31c26679542790c9c77b52cb7c6b_x32.cpkl
Collocation warning in process collocation settings: Bounds are not correctly handled


ValueError: Discretization settings are required when nnodes > 1.

## 2. Inspect the Structured States

Under the new **lean state vector** architecture:
* The legacy `.model` field is removed.
* States are stored in separate physical vectors: `q` (coordinates), `qd` (velocities), `qdd` (accelerations, optional), `tau` (actuator inputs/moments), `ext_forces`/`ext_torques` (external ground/sensor interactions).
* Let's inspect the `initial_guess_states` and `initial_guess_globals`.

In [3]:
ig_states = ocp.initial_guess_states
ig_globals = ocp.initial_guess_globals

print("--- Initial Guess States Structure ---")
print(f"Type: {type(ig_states)}")
print(f"q shape: {ig_states.states.q.shape if ig_states.states.q is not None else None}")
print(f"qd shape: {ig_states.states.qd.shape if ig_states.states.qd is not None else None}")
print(f"tau shape: {ig_states.states.tau.shape if ig_states.states.tau is not None else None}")
print(f"ext_forces shape: {ig_states.states.ext_forces.shape if ig_states.states.ext_forces is not None else None}")

print("\n--- Initial Guess Globals ---")
print(f"Globals fields: {ig_globals}")
print(f"dur (duration): {ig_globals.dur}")
print(f"speed: {ig_globals.speed}")
print(ig_states)

NameError: name 'ocp' is not defined

## 3. The Flat Optimization Vector `x`

IPOPT (the non-linear programming solver) expects a single flat 1D array of variables `x`.

BioSYM provides utility functions `states_dict_to_x` and `x_to_states_dict` to bidirectionally convert between the structured `StatesDict`/`Globals` objects and the flat variable vector `x`.

In [9]:
x0 = ocp.x0
print(f"Flat optimization vector x0 shape: {x0.shape}")
print(f"Number of state variables per node: {len(x0[:-2]) / ocp.settings['nnodes']}")
print(f"Appended global variables at the end of x0: {x0[-2:]}")

# Convert back to structured states
states_back, globals_back = collocation.utils.x_to_states_dict(x0, ocp.initial_guess_states, ocp.initial_guess_globals)
assert np.allclose(states_back.states.q, ig_states.states.q)
print("Bidirectional translation verified successfully!")

Flat optimization vector x0 shape: (7274,)
Number of state variables per node: 72.72
Appended global variables at the end of x0: [0.6 1.3]
Bidirectional translation verified successfully!


## 4. Inspect Constraints

Let's inspect the collocation constraints. BioSYM sets up constraints specified in the YAML configuration (like `dynamics` and `periodicity`).

We can evaluate constraint residuals at our current `x0` initial guess.

In [ ]:
print(f"Total number of constraints: {ocp.nlp.n_constraints}")
# Evaluate constraint residuals at x0
g_val = ocp.nlp.eval_g(x0)
print(f"Constraint residuals shape: {g_val.shape}")

# Let's inspect the individual constraints registered in the problem
for idx, constraint in enumerate(ocp.cons._constraints):
    c_name = constraint.__class__.__name__
    print(f"\nConstraint index {idx}: {c_name}")
    print(f"  Number of constraints: {constraint.get_n_constraints(ocp.model, ocp.settings)}")
    
    # We can evaluate a single constraint component using its internal evaluation
    if hasattr(constraint, "evaluate"):
        # Create a list of per-node StatesDicts for evaluation
        states_list = [states_back[i] for i in range(ocp.settings["nnodes"])]
        res = constraint.evaluate(states_list, globals_back, ocp.model, ocp.settings)
        print(f"  Residual shape: {res.shape}")
        print(f"  Max absolute residual: {np.max(np.abs(res)):.6g}")

## 5. Inspect Objectives

The total objective function is a weighted sum of individual objectives (e.g., joint angle tracking, efforts, and ground reaction forces tracking).

Let's evaluate them individually at the initial guess `x0`.

In [ ]:
# Evaluate total objective value at x0
f_val = ocp.nlp.eval_f(x0)
print(f"Total objective value at x0: {f_val:.6g}")

# Inspect individual objectives
for idx, obj in enumerate(ocp.objective._objectives):
    obj_info = obj._get_info() if hasattr(obj, "_get_info") else {"name": obj.__class__.__name__}
    obj_name = obj_info.get("name", "unknown")
    weight = obj_info.get("weight", 1.0)
    
    # Evaluate individual objective component
    val = obj.evaluate(states_back, globals_back, ocp.model, ocp.settings)
    weighted_val = val * weight
    print(f"\nObjective '{obj_name}' (weight {weight}):")
    print(f"  Raw value: {val:.6g}")
    print(f"  Weighted contribution: {weighted_val:.6g}")

## 6. JAX Automatic Differentiation & Sparsity

BioSYM leverages **JAX** to trace the constraint and objective functions, producing compiled gradients and Jacobians. 

Since each node's equations of motion and objectives only depend on variables at its local node (and occasionally adjacent nodes), the Jacobian matrix is extremely sparse. We can calculate this gradient and sparse Jacobian and plot the sparsity pattern.

In [ ]:
# Compute gradient of objective f
grad_f = ocp.nlp.eval_grad_f(x0)
print(f"Objective gradient shape: {grad_f.shape}")
print(f"Max gradient element: {np.max(np.abs(grad_f)):.6g}")

# Get Jacobian sparsity structure and evaluate Jacobian at x0
jac_rows, jac_cols = ocp.nlp.jacobianstructure()
jac_vals = ocp.nlp.eval_jac_g(x0)
print(f"Number of non-zero elements in Jacobian: {len(jac_vals)}")

# Build the sparse Jacobian matrix
from scipy.sparse import coo_matrix
J = coo_matrix((jac_vals, (jac_rows, jac_cols)), shape=(ocp.nlp.n_constraints, len(x0)))

# Plot sparsity pattern
plt.figure(figsize=(10, 8), dpi=100)
plt.spy(J, precision=1e-8, markersize=1, aspect="auto")
plt.title("Sparsity Pattern of Constraint Jacobian", fontsize=14)
plt.xlabel("Variables (x)", fontsize=12)
plt.ylabel("Constraints (g)", fontsize=12)
plt.grid(True, which='both', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

## 7. Run a Short Optimization Step

Let's configure IPOPT to run for a tiny number of iterations (e.g., 5 iterations) to observe the convergence behavior and iteration logging.

In [ ]:
# Configure IPOPT to run for only 5 iterations
ocp.nlp.add_option("max_iter", 5)

print("Starting short optimization solve (5 iterations)...")
x_sol, info = ocp.solve(visualize=False)

print("\n--- Solver Summary ---")
print(f"Status: {info.get('status_msg')}")
print(f"Final Objective: {info.get('obj_val'):.6g}")
print(f"Iterations: {info.get('iter')}")

## 8. Visualize the Joint Angle Trajectories

Let's convert our solved vector `x_sol` back into structured coordinates and plot some of the joint angles (like hip, knee, and ankle flexion) over the gait cycle time grid.

In [ ]:
sol_states, sol_globals = collocation.utils.x_to_states_dict(x_sol, ocp.initial_guess_states, ocp.initial_guess_globals)

# Extract coordinates names and values
q_vals = np.array(sol_states.states.q)  # shape (nnodes, n_coordinates)
time_grid = np.linspace(0, float(sol_globals.dur[0]), ocp.settings["nnodes"])

coord_names = list(ocp.model.coordinates["names"])
joints_to_plot = ["pelvis_ty", "hip_flexion_r", "knee_angle_r", "ankle_angle_r"]

plt.figure(figsize=(12, 8))
for i, joint in enumerate(joints_to_plot):
    if joint in coord_names:
        idx = coord_names.index(joint)
        plt.subplot(2, 2, i + 1)
        plt.plot(time_grid, q_vals[:, idx], 'b-', linewidth=2)
        plt.title(f"{joint.replace('_', ' ').title()}", fontsize=12)
        plt.xlabel("Time [s]")
        plt.ylabel("Angle [rad]" if "angle" in joint or "flexion" in joint else "Position [m]")
        plt.grid(True)

plt.suptitle("Joint Trajectories Over Half Gait Cycle (Walking)", fontsize=16, y=0.98)
plt.tight_layout()
plt.show()